# livecell — a tour

Everything below runs **locally**. Nothing is uploaded.

**Setup:** `deno jupyter --install`, then pick the **Deno** kernel (top right).

In [ ]:
// NOTE: the Deno kernel evaluates each cell as a SCRIPT, not a module — so
// `import.meta` does not exist here, and relative specifiers depend on the kernel's
// working directory. Locate the repo root explicitly, then import by absolute URL.

// `var` (not const/let) so re-running a cell doesn't throw "already declared".
function findUp(marker: string, from = Deno.cwd()): string {
  let dir = from;
  for (let i = 0; i < 8; i++) {
    try {
      Deno.statSync(`${dir}/${marker}`);
      return dir;
    } catch { /* keep walking up */ }
    const parent = dir.replace(/\/[^/]+$/, "");
    if (parent === dir) break;
    dir = parent;
  }
  throw new Error(`could not find ${marker} above ${from}`);
}

var root = findUp("deno.json");
var projectDir = `${root}/examples/demo-project`;

// Published: await import("jsr:@livecell/livecell")
var live = await import(`file://${root}/src/mod.ts`);

console.log("cwd    :", Deno.cwd());
console.log("root   :", root);
console.log("project:", projectDir);
console.log("port   :", live.start());

## 1 · Embed a real project

`mount` serves a folder; `embed` puts it in the cell. Click the button — it's a live page.

In [ ]:
live.mount("demo", projectDir);
live.embed("/m/demo/index.html", { height: 300, label: "demo project" });

## 2 · See what the page logged

`app.js` calls `console.log` on every click. livecell pipes that back into the notebook —
click the button above a few times, then run this cell.

In [ ]:
live.showLogs();

## 3 · Auto-reload while you edit

`watch` refreshes every embed when a file changes. Run this, then edit
`examples/demo-project/style.css` and watch the frame above update.

In [ ]:
live.watch(projectDir);
console.log("watching", projectDir, "— edit a file and the embeds reload");

## 4 · Diagrams that actually render

Notebook markdown cells **cannot** render Mermaid. Served from localhost, it just works.

In [ ]:
live.mermaid(`flowchart LR
  notes[concept notes] --> cell[runnable cell]
  cell --> app[live app on a port]
  app --> notes`, { height: 240 });

## 5 · Excalidraw — editable, and saved back to disk

`editable: true, save: true` turns the cell into a real editor. Move something, press
**Save**, and `demo-project/diagram.excalidraw.json` is rewritten.

In [ ]:
await live.excalidraw(`${projectDir}/diagram.excalidraw.json`, {
  editable: true,
  save: true,
  height: 420,
});

## 6 · A live playground

The only way a notebook can *show* how a layout behaves. Edit either pane.

In [ ]:
live.playground({
  html: `<div class="row">
  <div class="card">a</div>
  <div class="card">b</div>
  <div class="card">c</div>
</div>`,
  css: `.row { display: flex; justify-content: space-between; gap: 8px; }
.card { flex: 1; padding: 18px; background: #dbeafe; border-radius: 8px;
        font: 14px system-ui; text-align: center; }`,
  height: 380,
});

## 7 · Any server, any language

`serveCmd` spawns the process, waits until the port actually answers, then hands back the URL.
Works the same for `npm run dev`, `go run .`, `python3 -m http.server`.

In [ ]:
var url = await live.serveCmd("deno", [
  "eval",
  `Deno.serve({ port: 8931 }, () =>
     new Response("<body style='font:16px system-ui;padding:20px;background:#fff7ed'>" +
       "<h2 style='color:#c2410c'>Spawned server</h2><p>Started by a notebook cell.</p></body>",
       { headers: { "content-type": "text/html" } }));`,
], 8931);
live.embed(url, { height: 160, label: "deno server" });

## Tidy up

Stops the file server, all watchers, and every spawned process.

In [ ]:
await live.stopAll();